# Pipelines de procesamiento para el Dataset de Reseñas de productos y cuidado de la piel de Sephora

Este notebook construye un pipeline completo de limpieza y transformación para el dataset de `data/raw/`.

### Transformadores Personalizados:
Para solucionar los problemas de calidad de datos detectados en la fase del EDA, hemos construido una arquitectura de transformadores personalizados. Esto asegura que la limpieza, replicable y evite la fuga de información.

Los transformadores definidos para el modelo de Sephora son:

1. **`DropColumnsTransformer`**: Elimina variables que determinamos como inútiles desde el inicio (ej. el índice residual `Unnamed: 0`).
2. **`ColumnRenamerTransformer`**: Estandariza los nombres de las columnas que quedaron duplicadas con sufijos `_x` e `_y` tras la integración de los datasets de productos y reseñas.
3. **`UnknownToNaNTransformer`**: Estandariza strings como `'unknown'`, `'UNKNOWN'` o espacios en blanco, transformándolos en verdaderos valores nulos (`np.nan`).
4. **`DropHighMissingTransformer`**: Identifica y elimina automáticamente las columnas que superen el 90% de nulos (umbral definido tras detectar que variables como `variation_desc` estaban casi vacías).
5. **`OutlierCapper`**: Aplica la técnica de *Capping* (IQR) a las variables continuas para limitar valores extremos, excluyendo de forma inteligente las variables binarias y los IDs para no deformar su naturaleza.
6. **`DropZeroVarianceTransformer`**: Descarta variables numéricas que tengan varianza cero (el mismo valor en todas las filas), ya que no aportan poder predictivo a los modelos.
7. **`SmartImputerTransformer`**: Imputa valores faltantes según el tipo de dato (mediana para numéricas, moda para categóricas). Además, incorpora una regla: los nulos en la métrica `helpfulness` se rellenan con `0`, asumiendo que representan reseñas sin votos.
8. **`PriceCleanerTransformer`**: Transformador específico para limpiar las variables de precio (ej. `price_usd`), eliminando símbolos de moneda (`$`) o comas, y convirtiéndolas a formato numérico (`float`).
9. **`TextCleanerTransformer`**: Elimina saltos de línea (`\n`) y espacios residuales en las variables categóricas de texto utilizando `.strip()`.

In [64]:
# Se configura para que detecte y se ejecute Google Colab o local y definir los transformadores localmente
import os
import re
import sys
import glob
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn import set_config
from sklearn.base import BaseEstimator, TransformerMixin

# Se ve la lógica para que sea compatible entre Colab y el entorno local
try:
    import google.colab
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

archivos = []

if IN_COLAB:
    print('Google Colab detectado. Sube tu dataset unificado de Sephora cuando se te pida.')
    from google.colab import files
    uploaded = files.upload()
    if uploaded:
        data_filename = list(uploaded.keys())[0]
    else:
        raise FileNotFoundError('Debe subir el dataset en Colab antes de ejecutar el notebook.')
else:
    ruta_carpeta = "../data/raw/"
    archivos = glob.glob(os.path.join(ruta_carpeta, "*.csv"))

print('Usando ruta de datos:', data_filename)

# Se definen los transformadores personalizados

class DropColumnsTransformer(BaseEstimator, TransformerMixin):
    """Elimina columnas específicas del DataFrame que no aportan valor."""
    def __init__(self, columns_to_drop):
        self.columns_to_drop = columns_to_drop

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        # Se filtran solo las columnas que realmente existen para evitar errores
        cols = [col for col in self.columns_to_drop if col in X_copy.columns]
        return X_copy.drop(columns=cols)
    
    def set_output(self, transform=None):
        return self


class ColumnRenamerTransformer(BaseEstimator, TransformerMixin):
    """Renombra columnas cruzadas tras un merge (ej. price_usd_x -> price_usd)."""
    def __init__(self, rename_dict):
        self.rename_dict = rename_dict

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        return X_copy.rename(columns=self.rename_dict)
    
    def set_output(self, transform=None):
        return self


class UnknownToNaNTransformer(BaseEstimator, TransformerMixin):
    """Convierte el string 'unknown' o vacíos a un valor nulo real (np.nan)."""
    def fit(self, X, y=None):
        return self

    def transform(self, X):
        return X.replace(['unknown', 'UNKNOWN', 'Unknown', ''], np.nan)
    
    def set_output(self, transform=None):
        return self


class DropHighMissingTransformer(BaseEstimator, TransformerMixin):
    """Elimina las columnas que superen un umbral máximo de valores nulos."""
    def __init__(self, threshold=0.90): # Subimos el umbral al 90%
        self.threshold = threshold
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        pct_nulos = X.isnull().mean()
        self.cols_to_drop_ = pct_nulos[pct_nulos > self.threshold].index.tolist()
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [c for c in self.cols_to_drop_ if c in X_copy.columns]
        return X_copy.drop(columns=cols)
    
    def set_output(self, transform=None):
        return self


class OutlierCapper(BaseEstimator, TransformerMixin):
    """Aplica capping a los valores atípicos usando el método IQR."""
    def __init__(self, apply_capping=True):
        self.apply_capping = apply_capping
        self.bounds_ = {}

    def fit(self, X, y=None):
        if not self.apply_capping:
            return self
        
        # Excluimos IDs y binarias para no deformarlas
        columnas_a_excluir = ['author_id', 'product_id', 'brand_id', 'is_recommended', 
                              'limited_edition', 'new', 'online_only', 'out_of_stock', 'sephora_exclusive']
        
        num_cols = [col for col in X.select_dtypes(include=['number']).columns if col not in columnas_a_excluir]
        
        for col in num_cols: 
            Q1 = X[col].dropna().quantile(0.25)
            Q3 = X[col].dropna().quantile(0.75)
            IQR = Q3 - Q1
            self.bounds_[col] = (Q1 - 1.5 * IQR, Q3 + 1.5 * IQR)
        return self

    def transform(self, X):
        X_copy = X.copy()
        if not self.apply_capping:
            return X_copy
        for col, (lower, upper) in self.bounds_.items():
            if col in X_copy.columns:
                X_copy[col] = np.clip(X_copy[col], lower, upper)
        return X_copy
    
    def set_output(self, transform=None):
        return self
    
    def get_feature_names_out(self, input_features=None):
        return input_features


class DropZeroVarianceTransformer(BaseEstimator, TransformerMixin):
    """Elimina columnas numéricas con varianza igual a 0 (un solo valor constante)."""
    def __init__(self):
        self.cols_to_drop_ = []

    def fit(self, X, y=None):
        num_cols = X.select_dtypes(include=['number']).columns
        zero_variance = [col for col in num_cols if X[col].std() == 0]
        self.cols_to_drop_ = zero_variance
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols = [c for c in self.cols_to_drop_ if c in X_copy.columns]
        return X_copy.drop(columns=cols)
    
    def set_output(self, transform=None):
        return self
    
    def get_feature_names_out(self, input_features=None):
        return input_features


class SmartImputerTransformer(BaseEstimator, TransformerMixin):
    """Imputa nulos: Mediana para numéricas, Moda para categóricas."""
    def __init__(self, low_threshold=0.10):
        self.low_threshold = low_threshold
        self.cols_simples_ = []
        self.cols_complejas_ = []

    def fit(self, X, y=None):
        porcentaje_nulos = X.isnull().mean()
        for col in X.columns:
            pct = porcentaje_nulos[col]
            if 0 < pct <= self.low_threshold:
                self.cols_simples_.append(col)
            elif pct > self.low_threshold:
                self.cols_complejas_.append(col)
        print(f"SmartImputer - Simples (<10%): {self.cols_simples_}")
        print(f"SmartImputer - Complejas (>10%): {self.cols_complejas_}")
        return self

    def transform(self, X):
        X_copy = X.copy()
        todas_las_cols = self.cols_simples_ + self.cols_complejas_
        
        for col in todas_las_cols:
            if col in X_copy.columns:
                if col == 'helpfulness':
                    X_copy[col] = X_copy[col].fillna(0)
                elif pd.api.types.is_numeric_dtype(X_copy[col]):
                    X_copy[col] = X_copy[col].fillna(X_copy[col].median())
                else:
                    X_copy[col] = X_copy[col].fillna(X_copy[col].mode()[0])
        return X_copy
    
    def set_output(self, transform=None):
        return self


class PriceCleanerTransformer(BaseEstimator, TransformerMixin):
    """Limpia símbolos de moneda y convierte a formato numérico (float)."""
    def __init__(self, column='price_usd'):
        self.column = column

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        if self.column in X_copy.columns:
            # Primero nos aseguramos de que sea string
            X_copy[self.column] = X_copy[self.column].astype(str)
            X_copy[self.column] = X_copy[self.column].replace('[$,]', '', regex=True)
            X_copy[self.column] = pd.to_numeric(X_copy[self.column], errors='coerce')
        return X_copy
    
    def set_output(self, transform=None):
        return self


class TextCleanerTransformer(BaseEstimator, TransformerMixin):
    """Elimina espacios en blanco residuales y saltos de línea de variables tipo texto."""
    def __init__(self, columns=None):
        self.columns = columns

    def fit(self, X, y=None):
        return self

    def transform(self, X):
        X_copy = X.copy()
        cols_to_clean = self.columns if self.columns else X_copy.select_dtypes(include=['object']).columns
        for col in cols_to_clean:
            if col in X_copy.columns:
                X_copy[col] = (
                    X_copy[col]
                    .astype(str)
                    .str.lower()
                    .str.strip()
                    .str.replace(r'\s+', ' ', regex=True)   # múltiples espacios
                    .str.replace(r'[^a-z0-9\s]', '', regex=True)  # opcional: quitar símbolos
                )
        return X_copy
    
    def set_output(self, transform=None):
        return self

print('Entorno y transformadores definidos localmente para el dataset de Sephora.')

Usando ruta de datos: ../data/raw/
Entorno y transformadores definidos localmente para el dataset de Sephora.


### Observaciones de la Carga de Datos Crudos

Al visualizar las primeras filas del dataset unificado, podemos confirmar que la estrategia de ingesta fue exitosa:

1. **Captura Efectiva de Nulos:** Debido al parámetro `na_values`, los espacios vacíos y textos residuales (como `'unknown'` o `'null'`) fueron transformados automáticamente en valores `NaN` reales desde la carga (como se observa en las columnas `helpfulness` y `skin_tone`).
2. **Estado "Salvaje" Intencional:** Mantenemos las columnas residuales (como `Unnamed: 0`) y problemas de formato en los precios. No aplicaremos ninguna limpieza manual sobre este DataFrame.

**El objetivo de un Pipeline fuerte es la automatización total.** Al cargar los datos en su estado crudo, simulamos un entorno de producción real donde el sistema debe ser capaz de ingerir datos nuevos y procesarlos de principio a fin sin intervención humana.

In [ ]:
# Creamos una lista vacía donde almacenaremos cada DataFrame
df_list = []

# Se recorre cada archivo CSV, se lee con pandas y se agrega a la lista de DataFrames
for file in archivos:
    df_temp = pd.read_csv(
        file,
        # na_values indica que valores se deben interpretar como nulos
        na_values=['NULL', 'null', '', ' ', 'unknown', 'UNKNOWN']
    )
    df_list.append(df_temp)

# Se unen todos los DataFrames
df_raw = pd.concat(df_list, ignore_index=True) # ignore_index=True reinicia el índice para que no haya duplicados

print('Dimensiones de los datos crudos:', df_raw.shape)
print('\nMuestra de las primeras 5 filas:')
display(df_raw.head())

C:\Users\maria\AppData\Local\Temp\ipykernel_36192\1739561212.py:4: DtypeWarning: Columns (0: author_id) have mixed types. Specify dtype option on import or set low_memory=False.
  df_temp = pd.read_csv(


Dimensiones de los datos crudos: (1102905, 41)

Muestra de las primeras 5 filas:


,product_id,product_name,brand_id,brand_name,loves_count,rating,reviews,size,variation_type,variation_value,...,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,submission_time,review_text,review_title,skin_tone,eye_color,skin_type,hair_color
0,P473671,Fragrance Discovery Set,6342.0,19-69,6320.0,3.6364,11.0,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,P473668,La Habana Eau de Parfum,6342.0,19-69,3827.0,4.1538,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,P473662,Rainbow Bar Eau de Parfum,6342.0,19-69,3253.0,4.2500,16.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,P473660,Kasbah Eau de Parfum,6342.0,19-69,3018.0,4.4762,21.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,P473658,Purple Haze Eau de Parfum,6342.0,19-69,2691.0,3.2308,13.0,3.4 oz/ 100 mL,Size + Concentration + Formulation,3.4 oz/ 100 mL,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


## Asignación de los datos

### Definición de Rutas

Para que nuestro Pipeline funcione de manera segura y sin errores, debemos enseñarle a distinguir entre los diferentes tipos de datos presentes en el dataset de Sephora. Los tratamientos que recibirán son distintos:

* **Ruta Numérica (`variables_numericas`):** Incluye el precio y las métricas de interacción (`price_usd`, `rating`, etc.). Estas columnas pasarán por transformaciones estadísticas: cálculo de medianas para imputar nulos, cálculo de cuartiles para el *Capping* de outliers y limpieza de símbolos de moneda.
* **Ruta Categórica (`variables_categoricas`):** Incluye descriptores de texto como `brand_name_x` o `skin_type`. Estas columnas pasarán por limpieza de caracteres (espacios, saltos de línea) y una imputación de nulos basada en la moda (la categoría más frecuente).

Estas listas alimentarán a nuestro `ColumnTransformer`, el organizador principal que se encargará de dirigir cada columna por su ruta correspondiente en paralelo.

In [ ]:
df_raw = df_raw.dropna(axis=1, how='all')

variables_numericas = [
    'price_usd',
    'rating',
    'helpfulness',
    'total_feedback_count',
    'total_neg_feedback_count',
    'total_pos_feedback_count'
]

variables_categoricas = [
    'brand_name',
    'skin_type',
    'eye_color',
    'hair_color'
]

# validar existencia + excluir
variables_numericas = [
    c for c in variables_numericas
    if c in df_raw.columns and c not in exclude
]

variables_categoricas = [
    c for c in variables_categoricas
    if c in df_raw.columns and c not in exclude
]

print('Variables numéricas:', variables_numericas)
print('Variables categóricas:', variables_categoricas)

Variables numéricas: ['price_usd', 'rating', 'helpfulness', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count']
Variables categóricas: ['brand_name', 'skin_type', 'eye_color', 'hair_color']


## Diagrama interactivo

### Ejecución de los datos crudos a los datos listos

Al utilizar el método `.fit_transform()` sobre nuestro DataFrame crudo (`df_raw`), el notebook ejecuta automáticamente todas las operaciones en el orden exacto que diseñamos:

1. **Limpieza Estructural y Formato:** Elimina columnas basura, ajusta textos y corrige los formatos numéricos de los precios extrayendo solo el valor real.
2. **Gestión Inteligente de Nulos:** Aplica nuestras reglas (medias, modas y constantes) para imputar los valores faltantes sin perder valiosas reseñas.
3. **Preprocesamiento Matemático (Bifurcación):** El `ColumnTransformer` separa los caminos. Los números se ajustan con *Capping* (para contener outliers), se revisa su varianza y se estandarizan; mientras tanto, las categorías de texto se transforman en matrices binarias.

In [69]:
df_raw = df_raw.drop_duplicates(subset=['product_id', 'author_id'])

# 1. Ruta para números
num_pipe = Pipeline([
    ('capper', OutlierCapper(apply_capping=True)),
    ('zero_variance', DropZeroVarianceTransformer()),
    ('scaler', StandardScaler())
])

# 2. Ruta para textos
cat_pipe = Pipeline([
    ('onehot', OneHotEncoder(handle_unknown='ignore', sparse_output=False, max_categories=20))
])

('limpia_textos', TextCleanerTransformer())

# 3. El enrutador
preprocesador = ColumnTransformer(
    transformers=[
        ('num', num_pipe, variables_numericas),
        ('cat', cat_pipe, variables_categoricas)
    ], remainder='drop' # Esto descarta automáticamente todo lo que no esté en nuestras listas
)

# 4. Pipeline de Sephora
pipeline_ml_completo = Pipeline([
    
    ('drop_basura', DropColumnsTransformer(columns_to_drop=[
        'Unnamed: 0', 'author_id', 'product_id', 'brand_id', 
        'product_name_y', 'brand_name_y', 'price_usd_y', 'rating_y',
        'ingredients', 'highlights', 'variation_value', 'variation_type'
    ])),

    ('renombrar', ColumnRenamerTransformer({
        'price_usd_x': 'price_usd',
        'brand_name_x': 'brand_name'
    })),

    ('limpieza_unknowns', UnknownToNaNTransformer()),
    ('limpia_textos', TextCleanerTransformer()),
    ('limpia_precios', PriceCleanerTransformer(column='price_usd')),

    ('drop_high_nan', DropHighMissingTransformer(threshold=0.90)),
    ('smart_imputer', SmartImputerTransformer(low_threshold=0.10)),

    ('preprocesamiento', preprocesador)
])

# Y finalmente se muestra el diagrama interactivo
set_config(display='diagram')
pipeline_ml_completo

,"steps steps: list of tuplesList of (name of step, estimator) tuples that are to be chained insequential order. To be compatible with the scikit-learn API, all stepsmust define `fit`. All non-last steps must also define `transform`. See:ref:`Combining Estimators ` for more details.","[('drop_basura', ...), ('renombrar', ...), ...]"
,"transform_input transform_input: list of str, default=NoneThe names of the :term:`metadata` parameters that should be transformed by thepipeline before passing it to the step consuming it.This enables transforming some input arguments to ``fit`` (other than ``X``)to be transformed by the steps of the pipeline up to the step which requiresthem. Requirement is defined via :ref:`metadata routing `.For instance, this can be used to pass a validation set through the pipeline.You can only set this if metadata routing is enabled, which youcan enable using ``sklearn.set_config(enable_metadata_routing=True)``... versionadded:: 1.6",None
,"memory memory: str or object with the joblib.Memory interface, default=NoneUsed to cache the fitted transformers of the pipeline. The last stepwill never be cached, even if it is a transformer. By default, nocaching is performed. If a string is given, it is the path to thecaching directory. Enabling caching triggers a clone of the transformersbefore fitting. Therefore, the transformer instance given to thepipeline cannot be inspected directly. Use the attribute ``named_steps``or ``steps`` to inspect estimators within the pipeline. Caching thetransformers is advantageous when fitting is time consuming. See:ref:`sphx_glr_auto_examples_neighbors_plot_caching_nearest_neighbors.py`for an example on how to enable caching.",None
,"verbose verbose: bool, default=FalseIf True, the time elapsed while fitting each step will be printed as itis completed.",False
,columns_to_drop,"['Unnamed: 0', 'author_id', ...]"
,rename_dict,"{'brand_name_x': 'brand_name', 'price_usd_x': 'price_usd'}"
,columns,None
,column,'price_usd'
,threshold,0.9
,low_threshold,0.1
,"transformers transformers: list of tuplesList of (name, transformer, columns) tuples specifying thetransformer objects to be applied to subsets of the data.name : str Like in Pipeline and FeatureUnion, this allows the transformer and its parameters to be set using ``set_params`` and searched in grid search.transformer : {'drop', 'passthrough'} or estimator Estimator must support :term:`fit` and :term:`transform`. Special-cased strings 'drop' and 'passthrough' are accepted as well, to indicate to drop the columns or to pass them through untransformed, respectively.columns : str, array-like of str, int, array-like of int, array-like of bool, slice or callable Indexes the data on its second axis. Integers are interpreted as positional columns, while strings can reference DataFrame columns by name. A scalar string or int should be used where ``transformer`` expects X to be a 1d array-like (vector), otherwise a 2d array will be passed to the transformer. A callable is passed the input data `X` and can return any of the above. To select multiple columns by name or dtype, you can use :obj:`make_column_selector`.","[('num', ...), ('cat', ...)]"


### Transformación Exitosa y Verificación Final

Al ejecutar nuestro Pipeline sobre el dataset, los registros de salida y la matriz confirman que nuestra arquitectura procesó los datos perfectamente:

1. **Imputación Dinámica (`SmartImputer`):** El transformador identificó automáticamente el porcentaje de valores nulos en cada variable, clasificándolas en dos grupos:

- Variables simples (<10% de nulos): imputadas con estrategias conservadoras como la mediana o la moda.
- Variables complejas (>10% de nulos): tratadas con mayor cuidado debido a su alta proporción de datos faltantes.
2. **Escalado de Variables Numéricas (`StandardScaler`):** Las variables numéricas fueron transformadas para cumplir con una distribución estandarizada, centrada en cero y con desviación estándar.
Esto permite que todas las variables contribuyan de manera equilibrada al modelo, evitando sesgos por diferencias de escala entre atributos como precios o conteos de interacción.
3. **Expansión Binarizada (`One-Hot Encoding`):** Las variables categóricas fueron transformadas en variables binarias (0 y 1), representando la presencia de cada categoría. Este proceso aumentó la dimensionalidad del dataset, resultando en un total de **43 variables finales**, lo que permite que los modelos interpreten correctamente variables no numéricas.
4. **Limpieza de Nomenclatura:** Se aplicó una estandarización de nombres de variables para mejorar la interpretabilidad del dataset final, eliminando prefijos técnicos generados por el `ColumnTransformer`.
Esto permite trabajar con un conjunto de datos más legible y listo para modelamiento.


In [70]:
# fit_transform = primero aprende las transformaciones y luego las aplica
X_transformado = pipeline_ml_completo.fit_transform(df_raw)

# Se obtiene los nombre de las columnas resultantes
nombres_finales = pipeline_ml_completo.named_steps['preprocesamiento'].get_feature_names_out()

# Se convierte la matriz en un pandas
df_limpio = pd.DataFrame(X_transformado, columns=nombres_finales)

print(df_limpio.shape)

C:\Users\maria\AppData\Local\Temp\ipykernel_36192\76124920.py:237: Pandas4Warning: For backward compatibility, 'str' dtypes are included by select_dtypes when 'object' dtype is specified. This behavior is deprecated and will be removed in a future version. Explicitly pass 'str' to `include` to select them, or to `exclude` to remove them and silence this warning.
See https://pandas.pydata.org/docs/user_guide/migration-3-strings.html#string-migration-select-dtypes for details on how to write code that works with pandas 2 and 3.
  cols_to_clean = self.columns if self.columns else X_copy.select_dtypes(include=['object']).columns


SmartImputer - Simples (<10%): ['rating', 'total_feedback_count', 'total_neg_feedback_count', 'total_pos_feedback_count', 'submission_time', 'review_text']
SmartImputer - Complejas (>10%): ['is_recommended', 'helpfulness', 'review_title', 'skin_tone', 'eye_color', 'skin_type', 'hair_color']
(1097385, 43)


### Limpieza y Verificación de la Matriz Final

Como último paso de la transformación, aplicamos una limpieza a los nombres de las variables. Por defecto, el `ColumnTransformer` inyecta prefijos (como `num__` y `cat__`) para rastrear la ruta por la que pasó cada columna en el Pipeline. Al removerlos mediante comprensión de listas, recuperamos nombres limpios, legibles y listos para producción (por ejemplo, `brand_name_CLINIQUE` en lugar de `cat__brand_name_CLINIQUE`).

Al observar la tabla final, podemos confirmar visualmente el éxito de nuestra arquitectura de preprocesamiento:

* **Variables Numéricas Estandarizadas:** Columnas como `price_usd` y `rating` ya no muestran sus valores originales, sino valores decimales positivos y negativos. Esto confirma que el `StandardScaler` actuó correctamente, ajustando las distribuciones para que tengan media cero y desviación estándar de uno.
* **Expansión Binarizada (One-Hot Encoding):** El dataset pasó de muchas de variables a un total de **148 columnas**. Características de texto vitales para el negocio (como marcas, tonos de piel o colores de cabello) han sido separadas en columnas independientes compuestas exclusivamente por `0.0` y `1.0`.

In [38]:
# 1. Limpiamos los prefijos 'num__' y 'cat__' de los nombres de las columnas
nombres_limpios = [nombre.replace('num__', '').replace('cat__', '') for nombre in df_limpio.columns]

# 2. Le asignamos estos nombres limpios y bonitos a nuestro DataFrame final
df_limpio.columns = nombres_limpios

# 3. Imprimimos el resultado (mostrando solo una muestra para no saturar la pantalla)
print(f'Total de variables finales listas para el modelo: {len(df_limpio.columns)}\n')

print("Muestra de las columnas resultantes:")
# Imprimimos las primeras 20 para ver cómo quedaron (los números y las primeras categorías)
for i, nombre in enumerate(df_limpio.columns[:20]): 
    print(f'{i+1:02d}. {nombre}')

if len(df_limpio.columns) > 20:
    print(f"\n... y {len(df_limpio.columns) - 20} variables categóricas más generadas por el One-Hot Encoder.")

# Vemos la tabla final perfecta
display(df_limpio.head())

Total de variables finales listas para el modelo: 43

Muestra de las columnas resultantes:
01. price_usd
02. rating
03. helpfulness
04. total_feedback_count
05. total_neg_feedback_count
06. total_pos_feedback_count
07. brand_name_CLINIQUE
08. brand_name_Dermalogica
09. brand_name_Dr. Jart+
10. brand_name_Drunk Elephant
11. brand_name_Farmacy
12. brand_name_First Aid Beauty
13. brand_name_Glow Recipe
14. brand_name_LANEIGE
15. brand_name_Murad
16. brand_name_OLEHENRIKSEN
17. brand_name_Origins
18. brand_name_Peter Thomas Roth
19. brand_name_Sunday Riley
20. brand_name_Tatcha

... y 23 variables categóricas más generadas por el One-Hot Encoder.


,price_usd,rating,helpfulness,total_feedback_count,total_neg_feedback_count,total_pos_feedback_count,brand_name_CLINIQUE,brand_name_Dermalogica,brand_name_Dr. Jart+,brand_name_Drunk Elephant,...,eye_color_gray,eye_color_green,eye_color_hazel,hair_color_auburn,hair_color_black,hair_color_blonde,hair_color_brown,hair_color_brunette,hair_color_gray,hair_color_red
0,-0.404408,-0.872085,-0.838248,-0.723257,-0.527019,-0.660892,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2.558569,-0.285715,-0.838248,-0.723257,-0.527019,-0.660892,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
2,2.558569,-0.176691,-0.838248,-0.723257,-0.527019,-0.660892,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,2.558569,0.079662,-0.838248,-0.723257,-0.527019,-0.660892,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,2.558569,-1.331752,-0.838248,-0.723257,-0.527019,-0.660892,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


In [ ]:
import os

os.makedirs("data/processed", exist_ok=True)

df_limpio.to_csv("data/processed/sephora_limpio.csv", index=False)
df_limpio.to_excel("data/processed/sephora_limpio.xlsx", index=False)

### Resumen del Súper Pipeline de Sephora

El preprocesamiento de nuestros datos se ejecutó a través de un Pipeline que asegura reproducibilidad y evita la fuga de datos (Data Leakage). El flujo consta de los siguientes pasos:

- **Paso A (Limpieza Estructural):** Se eliminan columnas residuales producto del cruce de tablas (las terminadas en `_y`), identificadores únicos (IDs) y metadatos que no aportan valor predictivo para el modelo.
- **Paso B (Limpieza de Formatos):** Se estandarizan las cadenas de texto y se procesan los precios numéricos (`PriceCleanerTransformer`)-
- **Paso C (Gestión Avanzada de Nulos):** Se convierten valores de texto como "unknown" en nulos reales (`NaN`). Se descartan columnas irrecuperables (más del 90% de nulos) y se aplica un **Imputador Inteligente** que evalúa dinámicamente si usar imputación simple o compleja dependiendo de si los nulos superan el umbral del 10%.
- **Paso D (Enrutamiento Matemático - Numéricas):** A variables como el precio o el conteo de reseñas se les aplica *Capping* para contener valores atípicos (outliers), se filtran las de varianza cero, y se escalan con `StandardScaler` para equilibrar sus magnitudes.
- **Paso E (Enrutamiento Matemático - Categóricas):** Las variables de perfil del usuario y del producto (como `brand_name`, `primary_category`, `skin_type`, `skin_tone`, etc.) se expanden de forma binaria mediante `OneHotEncoder`.